# ContractRisk — comparación 02: corrección de interfaz M3

Comparación 01 se detuvo en el calentamiento: el adaptador intentaba .predict sobre funciones que devuelve la fábrica M3. No se evaluó ni guardó ninguno de los 22 casos. La corrección llama a esas funciones mediante un adaptador; también corrige el perfil M3 textual. Se conservan modelos, datos, prompts, semillas, límites, precisión y criterios. No se modificó M3 histórico.

Antes de cargar modelos se verifica el ZIP fallido, que no había respuestas evaluadas y que la configuración solo cambia de versión por la corrección técnica. Los errores futuros guardan archivo/línea/función sin mensaje sensible ni variables. Cerrar el error de interfaz no equivale a probar inferencia GPU de esta versión.


Esta ejecución recoge 22 entradas en cinco perfiles (110 filas si termina), además de calentamientos excluidos. No usa Gemini ni GEMINI_API_KEY, no modifica el presupuesto Drive y no necesita habilitar pagos. Requiere una sesión nueva de **GPU T4** y acceso a WANDB_API_KEY para la trazabilidad.

Perfiles: QA sin RAG adaptada, M3 textual reejecutado, integración visual FP16, integración visual con generador NF4 y ablación visual FP16 sin herramienta. Los perfiles textuales no reciben transcripciones gold; ante imágenes declaran que no las admiten y siguen contando en los 22 casos. No confundir esa limitación con un error de instalación. El clasificador M1 no se carga en este benchmark QA; su integración pública sigue pendiente.

Mismos casos y referencias preservadas, sin respuestas cacheadas. La pareja FP16/NF4 usa iguales modelos, instrucciones, recuperación, VLM y controles, salvo precisión del generador. Se conserva todo resultado, aunque sea incorrecto. Solo se permite la reparación de formato ya existente, no repetir buscando mejores respuestas.

Los diagnósticos automáticos son provisionales: **las notas del juez y el cumplimiento del dominio quedan pendientes**, no en cero ni aprobados. Recoger memoria/latencia no equivale a aceptar la cuantización. Una pasada ordenada por perfil no demuestra latencia estable ni causalidad absoluta; se informan resultados por modalidad y tiempos de carga por separado.

Ejecutar las celdas en orden y conservar ZIP/notebook con salidas. Si falla, ejecutar las celdas de inspección y descarga; no borrar evidencias ni repetir inferencias por cuenta propia. Esta ejecución no cambia main ni el Space público.


In [3]:
import os,sys,json,subprocess,tempfile
from pathlib import Path
from google.colab import userdata
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_comparison02_',dir='/content'))/'repo'
subprocess.run(['git','clone','--depth','1','--branch','Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
for commit in ['6ec8dcdf22433c632b5eb0e5539401c33b9c923d','71912f1c5755790381a8e39c8a67801bcbf9243d',
               '7afdda0212bf38ad1accd043a968db722569c618','ece58275c2a19abfb6bbd2fe4b507731232e9083']:
    subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',commit],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Commit:',COMMIT)
try:
    key=userdata.get('WANDB_API_KEY')
    if not key:raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY']=key
    del key
except Exception:
    raise RuntimeError('Habilita WANDB_API_KEY para este notebook. No pegues la clave.') from None
os.environ.setdefault('USE_TF','0')
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido. Conserva resultados y consulta '+logfile)


Commit: 8cbeb36ebdf3dde472920ffd7a655c0a5c5af38e


In [4]:
run_visible([sys.executable,'-m','pip','install','-r','final/requirements-comparison.txt'],'installation.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available(), 'Selecciona GPU T4'; assert 'T4' in torch.cuda.get_device_name(0), 'Se requiere T4'; print(torch.cuda.get_device_name(0))"],'gpu_check.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'tests_m3.log')


  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached sentence_transformers-3.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached chromadb-0.6.3-py3-none-any.whl.metadata (6.8 kB)
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached wandb-0.31.0-py3-none-manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_instrumentation_fastapi-0.66b2-py3-none-any.whl.metadata (2.2 kB)
  Using cached opentelemetry_exporter_otlp_proto_http-1.45.2-py3-none-any.whl.metadata (3.0 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached sentence_transformers-3.4.1-py3-none-any.whl (275 kB)
Using cached chromadb-0.6.3-py3-none-any.whl (611 kB)
Using cached peft-0.17.1-p

In [5]:
RESULTS=ROOT/'comparison02_results'
run_visible([sys.executable,'-u','final/run_common_collection.py','--output',str(RESULTS),'--require-wandb'],'comparison02.log')


Stage: validate_inputs_and_rules
Stage: gpu_and_package_preflight
Stage: wandb_connection
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from WANDB_API_KEY.
wandb: Currently logged in as: jcramonp (jcramonp-eanit) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: setting up run ve67u90y
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /content/contractrisk_comparison02_75b16v53/repo/wandb/run-20261119_022345-ve67u90y
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run clean-bee-19
wandb: ⭐️ View project at https://forge.coreweave.com/wandb/jcramonp-eanit/contractrisk-final
wandb: 🚀 View run at https://forge.coreweave.com/wandb/jcramonp-eanit/contractrisk-final/runs/ve67u90y
Stage: nf4_kernel_preflight
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (16) is not aligned for fast kernel with blocksize=64, falling b

## Inspección y conservación

`collection_completed_semantic_evaluation_pending` indica que terminó la recolección, no que se aprobó calidad o eficiencia. Si aparece `failed_partial_evidence_retained`, enviar el parcial para diagnóstico. No interpretar notas ausentes como aprobaciones.


In [6]:
RESULTS=ROOT/'comparison02_results'
for name in ['metadata.json','scorecards_candidate.json']:
    path=RESULTS/name
    if path.exists():print(name,path.read_text(encoding='utf-8'))


metadata.json {
  "status": "collection_completed_semantic_evaluation_pending",
  "purpose": "generation_and_measurements_only",
  "judge_used": false,
  "judge_validated_for_final": false,
  "quality_accepted": false,
  "efficiency_accepted": false,
  "started_utc": "2026-11-19T02:23:38.904135+00:00",
  "profiles_completed": [
    "qa_no_rag_fp16",
    "m3_text_fp16",
    "integrated_fp16",
    "integrated_nf4",
    "integrated_no_tool_fp16"
  ],
  "wandb_status": "finished_check_dashboard",
  "stage": "integrated_no_tool_fp16:automatic_diagnostics_no_judge",
  "recovery": {
    "archive_sha256": "7863af2a4126fd5f445d1912a94210538a018d2727516f7b65ae204f61473a11",
    "previous_evaluated_rows": 0,
    "settings_changed_except_version": false,
    "resampling_valid_answers": false,
    "repair": "M3 factory functions are called directly through a predict adapter, not treated as objects with predict"
  },
  "domain_rules_sha256": "56cefc9bf3329d758a2d382868486b054f63c01359e3b24befd4faa1c

In [7]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS/'notebook_commit.txt').write_text(COMMIT+'\n',encoding='utf-8')
for path in ROOT.glob('*.log'):shutil.copy2(path,RESULTS/path.name)
for name in ['requirements-comparison.txt','requirements-integration.txt','efficiency_policy.json']:
    shutil.copy2(ROOT/'final'/name,RESULTS/name)
package=shutil.make_archive(str(ROOT/'contractrisk_final_comparacion02_resultados'),'zip',RESULTS)
files.download(package)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>